# 1.2 Growth and strength: is there a link?
A side analysis, outside the app models. It asks one question: **do samples that grow faster or fuller also end up stronger?**

Two datasets can speak to it, each in its own way:
- **d005** (*T. versicolor*, Elsacker 2019): the photographed Petri dishes and the crushed blocks are *different objects*, but they share the same recipe (20% fibre, 70% water, 10% spawn, 16 days at 28 °C). So we compare **per substrate**: one dish's growth against the average strength of that substrate's blocks. Only unpressed blocks (FER-001) are used. 6 substrates have both.
- **d003** (*G. tsugae*, Gomaa 2024): the same printed specimen was photographed while growing and then crushed. So we compare **per specimen**. 10 specimens have both.

Every comparison is shown with its **standard error (SE)**: roughly, how much the number would wobble if the experiment were repeated. With 6 or 10 points, the SE matters as much as the number itself.

**Benchmarks**
1. *Average baseline*: predict every substrate's strength as the d005 average. Growth must beat this to be useful.
2. *Density law (literature)*: in foams, strength rises with density to a power, about 1.5 (Gibson & Ashby). Does growth add anything that density does not already explain?
3. *Growth-time optimum (Elsacker 2021)*: strength peaks at some growth time, then falls. Checked as a direction only.
4. *d003*: does the d005 direction (if any) also hold in a different species and a printed shape?

Nothing here is written to disk; it only reads files.

In [ ]:
### Cell 1. Imports and paths
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from mycodemo.config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR

EXTRACTED = INTERIM_DATA_DIR / "extracted"
D003 = EXTERNAL_DATA_DIR / "d003_gomaa2024"
rng = np.random.default_rng(42)   # fixed seed so the SEs are the same each run
N_BOOT = 2000

### Helper functions
- **Spearman's rho**: a correlation on *ranks* (1st, 2nd, 3rd...). +1 = the order matches perfectly, 0 = no relation, −1 = opposite order. Ranks are used because with so few points one extreme value would dominate an ordinary correlation.
- **Bootstrap SE**: redo the calculation many times on random re-draws of the same points (some points twice, some left out); the spread of the answers is the SE.
- **Permutation p-value**: shuffle one column many times and count how often a shuffled rho is as strong as the real one. A small p (e.g. below 0.05) means "unlikely to be luck". This notebook runs several such tests on the same few points, so one small p is a hint, not proof.
- A rho counts as having a **clear direction** here only if it is more than 2 SE away from 0.

In [ ]:
def spearman(x, y):
    x, y = pd.Series(x).rank(), pd.Series(y).rank()
    return float(np.corrcoef(x, y)[0, 1])

def boot_se(x, y, stat=spearman, n=N_BOOT):
    x, y = np.asarray(x, float), np.asarray(y, float)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(x), len(x))
        if len(set(i)) < 3:
            continue
        v = stat(x[i], y[i])
        if np.isfinite(v):
            vals.append(v)
    return float(np.std(vals))

def perm_p(x, y, stat=spearman, n=N_BOOT):
    real = abs(stat(x, y))
    y = np.asarray(y, float)
    hits = sum(abs(stat(x, rng.permutation(y))) >= real for _ in range(n))
    return (hits + 1) / (n + 1)

def report(name, x, y):
    r, se = spearman(x, y), boot_se(x, y)
    print(f"{name}: rho = {r:+.2f} ± {se:.2f} (SE), p = {perm_p(x, y):.3f}, n = {len(x)}")
    return r, se

## Part A. d005 by substrate
### Cell A1. Growth per dish
Coverage comes from `coverage_auto.csv` (made by notebook 3.0). Two growth numbers per dish:
- `cov_day9`: fraction of the dish covered on day 9 (the last day all 6 dishes have; read between photos by straight-line interpolation).
- `day_to_50`: the day the dish passed 50% cover (empty if it never did).

Straw (S) and flax dust (FD) are left out: their dishes barely grew and there are no unpressed blocks to match.

In [ ]:
cov = pd.read_csv(INTERIM_DATA_DIR / "coverage_auto.csv")
DISH_TO_SUBSTRATE = {   # coverage series -> substrate_id (checked against image.csv)
    "d005-H": "SUB-001",     # hemp
    "d005-Flax": "SUB-003",  # flax shives
    "d005-FLT": "SUB-005",   # flax tow, treated
    "d005-FLU": "SUB-006",   # flax tow, untreated
    "d005-FSW": "SUB-008",   # flax waste
    "d005-W": "SUB-009",     # softwood
}

def cover_on(g, day):
    g = g.dropna(subset=["coverage"]).sort_values("day")
    return float(np.interp(day, g["day"], g["coverage"]))

def day_to(g, level=0.5):
    g = g.dropna(subset=["coverage"]).sort_values("day").reset_index(drop=True)
    above = g.index[g["coverage"] >= level]
    if len(above) == 0:
        return np.nan
    j = above[0]
    if j == 0:
        return float(g.loc[0, "day"])
    a, b = g.loc[j - 1], g.loc[j]
    return float(a.day + (level - a.coverage) / (b.coverage - a.coverage) * (b.day - a.day))

growth = pd.DataFrame([
    {"series": k, "substrate_id": v,
     "cov_day9": cover_on(cov[cov.series_id == k], 9),
     "day_to_50": day_to(cov[cov.series_id == k])}
    for k, v in DISH_TO_SUBSTRATE.items()
])
growth.round(2)

### Cell A2. Strength and density per substrate
Unpressed blocks only (fermentation FER-001). `se` is the standard error of the substrate mean: SD ÷ √n.

In [ ]:
sample = pd.read_csv(EXTRACTED / "sample.csv")
meas = pd.read_csv(EXTRACTED / "measurement.csv")
STRESS, DENS = "compressive_stress_at_10pct_strain_MPa", "dry_density_kg_m3"

blocks = (meas[meas.dataset_id == "d005"]
          .merge(sample[["sample_id", "substrate_id", "fermentation_id"]], on="sample_id"))
blocks = blocks[(blocks.fermentation_id == "FER-001")
          & blocks[STRESS].notna()
                & blocks.substrate_id.isin(DISH_TO_SUBSTRATE.values())]

per_sub = blocks.groupby("substrate_id").agg(
    n=("sample_id", "size"), stress=(STRESS, "mean"), stress_sd=(STRESS, "std"),
    density=(DENS, "mean"))
per_sub["stress_se"] = per_sub.stress_sd / np.sqrt(per_sub.n)
a = growth.merge(per_sub.reset_index(), on="substrate_id")
print(f"{len(blocks)} blocks, {len(a)} substrates")
a.round(3)

### Cell A3. Picture
Each dot is one substrate; the vertical bar is ± 1 SE of its mean strength.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for axis, col, label in [(ax[0], "cov_day9", "dish cover on day 9"), (ax[1], "density", "block density, kg/m³")]:
    axis.errorbar(a[col], a.stress, yerr=a.stress_se, fmt="o", capsize=3)
    for _, r in a.iterrows():
        axis.annotate(r.series.replace("d005-", ""), (r[col], r.stress), textcoords="offset points", xytext=(5, 3))
    axis.set_xlabel(label); axis.set_ylabel("stress at 10% strain, MPa")
ax[0].set_title("Growth vs strength"); ax[1].set_title("Density vs strength")
plt.tight_layout(); plt.show()

### Cell A4. Rank correlations
Growth vs strength, density vs strength, and growth vs density. If growth and density move together, a growth–strength link could just be density in disguise.

In [ ]:
r_growth_d005 = report("cover day 9 vs strength", a.cov_day9, a.stress)
ok = a.day_to_50.notna()
report("day to 50% vs strength (lower = faster)", a.day_to_50[ok], a.stress[ok])
report("density vs strength", a.density, a.stress)
report("cover day 9 vs density", a.cov_day9, a.density)

### Benchmark 1. Average baseline, leave one substrate out
For each substrate in turn: hide it, fit on the other 5, predict the hidden one. Error is on a log scale, shown as a **fold error** (1.0 = perfect, 1.5 = typically off by half again). The ± is the SE over the 6 folds.

Models compared: the average (benchmark 1), a straight line on growth, the density power law (benchmark 2), and density plus growth.

In [ ]:
y = np.log(a.stress.values)
X_options = {
    "average (benchmark 1)": np.empty((len(a), 0)),
    "growth: cover day 9": a[["cov_day9"]].values,
    "density law (benchmark 2)": np.log(a[["density"]].values),
    "density + growth": np.column_stack([np.log(a.density), a.cov_day9]),
}

def loo_fold_error(X, y):
    errs = []
    for i in range(len(y)):
        keep = np.arange(len(y)) != i
        A = np.column_stack([np.ones(keep.sum()), X[keep]])
        coef, *_ = np.linalg.lstsq(A, y[keep], rcond=None)
        pred = np.r_[1, X[i]] @ coef
        errs.append(abs(pred - y[i]))
    errs = np.array(errs)
    return np.exp(errs.mean()), np.exp(errs.mean()) * errs.std(ddof=1) / np.sqrt(len(errs))

loo = pd.DataFrame([(k, *loo_fold_error(X, y)) for k, X in X_options.items()],
                   columns=["model", "fold_error", "se"]).round(2)
loo

### Benchmark 2. The density law itself
Fit log(strength) = a + b·log(density) on the individual d005 blocks. Gibson & Ashby's foam law predicts b ≈ 1.5 for crushing an open-cell foam; our pooled data across labs gave about 1.7. The SE of b says whether d005 agrees.

Then: does growth explain the strength that density leaves unexplained (the **residual**)?

In [ ]:
xb, yb = np.log(blocks[DENS].values), np.log(blocks[STRESS].values)
A = np.column_stack([np.ones_like(xb), xb])
coef, res, *_ = np.linalg.lstsq(A, yb, rcond=None)
resid = yb - A @ coef
s2 = resid @ resid / (len(yb) - 2)
b_se = np.sqrt(s2 * np.linalg.inv(A.T @ A)[1, 1])
print(f"d005 blocks: strength ~ density^{coef[1]:.2f} ± {b_se:.2f} (SE), n = {len(yb)} blocks")
print(f"Literature: 1.5 (Gibson & Ashby, open-cell foam); our pooled labs: about 1.7")

a["resid"] = np.log(a.stress) - (coef[0] + coef[1] * np.log(a.density))
report("cover day 9 vs strength left over after density", a.cov_day9, a.resid)

### Benchmark 3. Growth-time optimum (Elsacker 2021)
Every d005 block grew for the same 16 days, and d003 for about 80 hours, so neither dataset varies growth *time*. This benchmark cannot be tested here; it stays a literature rule in the app.

## Part B. d003 by specimen
### Cell B1. Growth per specimen: brightening index
d003 mycelium is pale blue, so colour thresholds fail (see notebook 3.0). Instead, each segment photo is turned to grey and its average brightness is taken at hour 0 and hour 80. **Brightening** = hour 80 minus hour 0, averaged over the specimen's segments. It is a crude growth index, not a % cover. Takes a minute (730 small images).

In [ ]:
rows = []
for p in sorted((D003 / "00_curve_segments_images").glob("S*_F*_*_*.jpg")):
    s, f, seg, hour = map(int, re.match(r"S(\d+)_F(\d+)_(\d+)_(\d+)", p.stem).groups())
    if hour in (0, 80):
        grey = np.asarray(Image.open(p).convert("L"), float)
        rows.append({"specimen": s, "segment": seg, "hour": hour, "brightness": grey.mean()})
img = pd.DataFrame(rows).pivot_table(index=["specimen", "segment"], columns="hour", values="brightness").dropna()
img["gain"] = img[80] - img[0]
d003_growth = img.groupby("specimen").gain.agg(brightening="mean", sd="std", n_segments="count")
d003_growth["se"] = d003_growth.sd / np.sqrt(d003_growth.n_segments)
d003_growth.round(1)

### Cell B2. Strength per specimen: stress at 5% strain
All 17 tests reach 5% strain (only 4 reach 10%), so 5% is used. Stress = force ÷ outer footprint (length × width), the same basis as a solid block. Strain = deformation ÷ height.

In [ ]:
params = pd.read_csv(D003 / "01_structural_dataset" / "00_samples_parameters.csv", encoding="utf-8-sig")
params = params.rename(columns={"Specimen": "specimen", "Length/mm": "L", "Width/mm": "W", "Height/mm": "H"})

def stress_at(spec, strain=0.05):
    curve = pd.read_csv(D003 / "01_structural_dataset" / "00_samples_compression_test" / f"{spec}.csv",
                        encoding="utf-8-sig")
    curve.columns = ["deformation_mm", "force_N"]
    p = params.set_index("specimen").loc[spec]
    reached = curve[curve.deformation_mm >= strain * p.H]
    if reached.empty:
        return np.nan
    return reached.force_N.iloc[0] / (p.L * p.W)   # N/mm² = MPa

params["stress_5pct_MPa"] = [stress_at(s) for s in params.specimen]
b = d003_growth.join(params.set_index("specimen")["stress_5pct_MPa"], how="inner").reset_index()
print(f"{len(b)} specimens with both photos and a compression test")
b.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.errorbar(b.brightening, b.stress_5pct_MPa, xerr=b.se.fillna(0), fmt="o", capsize=3)
for _, r in b.iterrows():
    ax.annotate(int(r.specimen), (r.brightening, r.stress_5pct_MPa), textcoords="offset points", xytext=(5, 3))
ax.set_xlabel("brightening, hour 0 to 80 (grey levels)"); ax.set_ylabel("stress at 5% strain, MPa")
ax.set_title("d003: growth vs strength, same specimen"); plt.tight_layout(); plt.show()
r_growth_d003 = report("d003 brightening vs stress at 5%", b.brightening, b.stress_5pct_MPa)

## Summary: growth vs the benchmarks
Read the table with the SE in mind: a rho of +0.6 ± 0.3 from 6 points is a hint, not a finding. Benchmark 4 asks only whether the **sign** agrees between d005 and d003, since species, timescale, shape and strain level all differ.

In [ ]:
def clear(r):
    return abs(r[0]) > 2 * r[1]

def direction(r1, r2):
    if not (clear(r1) and clear(r2)):
        return "at least one has no clear direction"
    return "same direction" if np.sign(r1[0]) == np.sign(r2[0]) else "opposite directions"

summary = pd.DataFrame([
    ["1 Average baseline", "fold error, growth vs average",
     f"{loo.set_index('model').loc['growth: cover day 9','fold_error']:.2f} vs {loo.set_index('model').loc['average (benchmark 1)','fold_error']:.2f}"],
    ["2 Density law", "fold error, density+growth vs density",
     f"{loo.set_index('model').loc['density + growth','fold_error']:.2f} vs {loo.set_index('model').loc['density law (benchmark 2)','fold_error']:.2f}"],
    ["2 Density law", "d005 exponent vs literature 1.5", f"{coef[1]:.2f} ± {b_se:.2f}"],
    ["3 Growth-time optimum", "testable here?", "no (growth time fixed in both datasets)"],
    ["4 d003", "growth-strength rho, d005 vs d003",
     f"{r_growth_d005[0]:+.2f} ± {r_growth_d005[1]:.2f} vs {r_growth_d003[0]:+.2f} ± {r_growth_d003[1]:.2f} ({direction(r_growth_d005, r_growth_d003)})"],
], columns=["benchmark", "check", "result"])
summary

**Limits.** d005: one dish per substrate and 2 to 3 blocks each, dishes and blocks are different objects, and fibre type and density move together, so any growth effect cannot be separated from "this substrate is just better". d003: brightening is a crude index that also picks up background and lighting; no density; stress at 5% not 10%. None of this feeds M3 or the app. It is a side story about whether growth seen in photos says anything about strength.